<img src='http://hilpisch.com/taim_logo.png' width="350px" align="right">

# Artificial Intelligence in Finance

## Convolutional Neural Networks

Dr Yves J Hilpisch | The AI Machine

http://aimachine.io | http://twitter.com/dyjh

## Imports

In [1]:
import os

import numpy as np
import pandas as pd
from pylab import plt, mpl

plt.style.use('seaborn-v0_8')
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '5'

In [2]:
import pandas as pd
aiif_eikon_eod_data = pd.read_csv('../../data/playground/aiif_eikon_eod_data.csv', sep=',', quotechar='"')
aiif_eikon_eod_data

,Date,AAPL.O,MSFT.O,INTC.O,AMZN.O,GS.N,SPY,.SPX,.VIX,EUR=,XAU=,GDX,GLD
0,2010-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.4323,1096.3500,NaN,NaN
1,2010-01-04,30.572827,30.950,20.88,133.90,173.08,113.33,1132.99,20.04,1.4411,1120.0000,47.71,109.80
2,2010-01-05,30.625684,30.960,20.87,134.69,176.14,113.63,1136.52,19.35,1.4368,1118.6500,48.17,109.70
3,2010-01-06,30.138541,30.770,20.80,132.25,174.26,113.71,1137.14,19.16,1.4412,1138.5000,49.34,111.51
4,2010-01-07,30.082827,30.452,20.60,130.00,177.67,114.19,1141.69,19.06,1.4318,1131.9000,49.10,110.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2604,2019-12-26,289.910000,158.670,59.82,1868.77,231.21,322.94,3239.91,12.65,1.1096,1511.2979,29.08,142.38
2605,2019-12-27,289.800000,158.960,60.08,1869.80,230.66,322.86,3240.02,13.43,1.1175,1510.4167,28.87,142.33
2606,2019-12-30,291.520000,157.590,59.62,1846.89,229.80,321.08,3221.29,14.82,1.1197,1515.1230,29.49,142.63
2607,2019-12-31,293.650000,157.700,59.85,1847.84,229.93,321.86,3230.78,13.78,1.1210,1517.0100,29.28,142.90


In [3]:
symbol = 'EUR='

In [5]:
data = aiif_eikon_eod_data

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2609 entries, 0 to 2608
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2609 non-null   str    
 1   AAPL.O  2516 non-null   float64
 2   MSFT.O  2516 non-null   float64
 3   INTC.O  2516 non-null   float64
 4   AMZN.O  2516 non-null   float64
 5   GS.N    2516 non-null   float64
 6   SPY     2516 non-null   float64
 7   .SPX    2516 non-null   float64
 8   .VIX    2516 non-null   float64
 9   EUR=    2609 non-null   float64
 10  XAU=    2602 non-null   float64
 11  GDX     2516 non-null   float64
 12  GLD     2516 non-null   float64
dtypes: float64(12), str(1)
memory usage: 290.6 KB


In [7]:
lags = 5

In [8]:
features = [symbol, 'r', 'd', 'sma', 'min', 'max', 'mom', 'vol']

In [9]:
def add_lags(data, symbol, lags, window=20, features=features):
    cols = []
    df = data.copy()
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift(1))
    df['sma'] = df[symbol].rolling(window).mean()
    df['min'] = df[symbol].rolling(window).min()
    df['max'] = df[symbol].rolling(window).max()
    df['mom'] = df['r'].rolling(window).mean()
    df['vol'] = df['r'].rolling(window).std()
    df.dropna(inplace=True)
    df['d'] = np.where(df['r'] > 0, 1, 0)
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

In [10]:
data, cols = add_lags(data, symbol, lags, window=20, features=features)

TypeError: operation 'truediv' not supported for dtype 'str' with dtype 'str'

In [ ]:
split = int(len(data) * 0.8)

In [ ]:
train = data.iloc[:split].copy()

In [ ]:
mu, std = train[cols].mean(), train[cols].std()

In [ ]:
train[cols] = (train[cols] - mu) / std

In [ ]:
test = data.iloc[split:].copy()

In [ ]:
test[cols] = (test[cols] - mu) / std

In [ ]:
import random
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten

In [ ]:
def set_seeds(seed=1000):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
set_seeds()
model = Sequential()
model.add(Conv1D(filters=96, kernel_size=5, activation='relu',
                 input_shape=(len(cols), 1)))
model.add(Flatten())
model.add(Dense(10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
%%time
model.fit(np.atleast_3d(train[cols]), train['d'],
          epochs=60, batch_size=48, verbose=False,
          validation_split=0.15, shuffle=False)

In [ ]:
res = pd.DataFrame(model.history.history)

In [ ]:
res.tail(3)

In [ ]:
res.plot(figsize=(10, 6));

In [ ]:
model.evaluate(np.atleast_3d(test[cols]), test['d'])

In [ ]:
test['p'] = np.where(model.predict(np.atleast_3d(test[cols])) > 0.5, 1, 0)

In [ ]:
test['p'] = np.where(test['p'] > 0, 1, -1)

In [ ]:
test['p'].value_counts()

In [ ]:
(test['p'].diff() != 0).sum()

In [ ]:
test['s'] = test['p'] * test['r']

In [ ]:
ptc = 0.00012 / test[symbol]

In [ ]:
test['s_'] = np.where(test['p'] != 0, test['s'] - ptc, test['s'])

In [ ]:
test[['r', 's', 's_']].sum().apply(np.exp)

In [ ]:
test[['r', 's', 's_']].cumsum().apply(np.exp).plot(figsize=(10, 6));

<img src='http://hilpisch.com/taim_logo.png' width="350px" align="right">